# 04 — Proteome-Scale Analysis: Reproduce Figures 7 and 8

Paper-faithful reproduction (Cazals & Sarti 2025):
- **Integer pLDDT discretisation** (pipeline default) — pLDDT rounded to integers on [0, 100] before filtration, matching AlphaFold's own discretisation and the paper's discretised treatment.
- **No F1-only restriction** for Figure 7 — all 23 391 fragments included.
- **n ≥ 200 filter** applied only for the Figure 8 fragmentation candidates, matching the paper's three-way query.
- **Batched sublevel-set filtration** (Elder Rule) — residues sharing an integer pLDDT level are inserted together as one batch, giving a piecewise-constant N_cc curve.

Targets:
- Figure 7: Pearson r(f⁺_cp, H_p) ≈ 0.97 across the H. sapiens proteome.
- Figure 8: filter (n ≥ 200, H_p ≥ 0.25, PLM ≥ 3) at t_p = 0.025 yields ~86 structures.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("../src").resolve()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import pearsonr, fisher_exact

from filtration import build_pd_and_ncc
from statistics import persistence_entropy, f_cp_plus, ncc_max as stat_ncc_max
from plm import plm as compute_plm, plm_multi
from plots import figure7, figure8_scatter

OUT = pathlib.Path("../data/results")
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the full proteome parquet (all 23 391 fragments, paper-faithful filtration)
df = pd.read_parquet("../data/results/proteome_full.parquet")
print(f"Total fragments: {len(df):,}")
print(f"\nDistribution of key statistics:")
print(df[["f_cp_plus", "H_p", "ncc_max", "PLM"]].describe().round(4))


In [ ]:
# Figure 7 — scatter f+_cp vs H_p over ALL 23 391 fragments (no length filter)
r7, p7 = pearsonr(df["f_cp_plus"], df["H_p"])
print(f"Pearson r = {r7:.4f}  (p ≈ {p7:.1e})   paper target ≈ 0.97")

figure7(df, out_path=OUT / "figure7_full.png")


In [ ]:
# Figure 8A — 3D scatter; fragmentation candidates use n>=200, H_p>=0.25, PLM>=3
mask = (df["n_residues"] >= 200) & (df["H_p"] >= 0.25) & (df["PLM"] >= 3)
cands = df[mask]
print(f"Fragmentation candidates (n≥200, H_p≥0.25, PLM≥3): {len(cands)}  [paper target: ~86]")

df_n200 = df[df["n_residues"] >= 200]
figure8_scatter(df_n200, out_path=OUT / "figure8a_scatter.png",
                calibrated_mask=mask[df_n200.index])


In [ ]:
# t_p sensitivity ablation
abl = pd.read_parquet("../data/results/proteome_full_tp_ablation.parquet")
print("t_p sensitivity (n≥200, H_p≥0.25, PLM≥3):")
for col, tp in [("PLM_020", 0.020), ("PLM_025", 0.025), ("PLM_030", 0.030)]:
    n = ((abl["n_residues"] >= 200) & (abl["H_p"] >= 0.25) & (abl[col] >= 3)).sum()
    print(f"  t_p = {tp}: {n} candidates")


In [ ]:
# Q1 x Q4 cross-correlation — arity map and enrichment
arity = pd.read_parquet("../data/results/proteome_full_arity.parquet")
merged = df.merge(arity[["uniprot_id", "filename", "arity_25", "arity_75"]],
                  on=["uniprot_id", "filename"])

cands_m = merged[(merged["n_residues"] >= 200) & (merged["H_p"] >= 0.25) & (merged["PLM"] >= 3)]
print(f"Candidates with arity: {len(cands_m)}")

# Centroid
a25c = int(round(cands_m["arity_25"].median()))
a75c = int(round(cands_m["arity_75"].median()))
print(f"Centroid: a25={a25c}, a75={a75c}")

# Fisher exact on centroid bin
in_bin_c  = ((cands_m["arity_25"] == a25c) & (cands_m["arity_75"] == a75c)).sum()
in_bin_bg = ((merged["arity_25"]  == a25c) & (merged["arity_75"]  == a75c)).sum()
n_c, n_bg = len(cands_m), len(merged)
table = [[in_bin_c, n_c - in_bin_c],
         [in_bin_bg - in_bin_c, n_bg - n_c - (in_bin_bg - in_bin_c)]]
_, pval = fisher_exact(table, alternative="greater")
enrichment = (in_bin_c / n_c) / (in_bin_bg / n_bg)
print(f"Enrichment at ({a25c},{a75c}): {enrichment:.2f}x  p={pval:.2e}")
